# Readify

## Project Overview

This project builds a beginner-friendly Retrieval-Augmented Generation (RAG) system that recommends books and songs based on a user's mood, theme, or preference.

The system uses two datasets:
- a Goodreads books dataset
- a Spotify songs dataset

The retrieval side of the project focuses on cleaning the data, creating searchable text descriptions, generating embeddings, and retrieving the most relevant books and songs for a user query.

**Team Members:** Sarbotam and Niveetha

### Setup: Installing Packages

Before you run the code below, make sure you complete all of the pre-requisites mentioned in class. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ollama
import warnings; warnings.filterwarnings("ignore")

---
## Part 1: Loading and setting up datasets

We load the two datasets that will be used in the project:
- Goodreads books
- Spotify songs

Both datasets actively feed into the RAG pipeline.

In [2]:
df_books = pd.read_csv("goodreads_top100_from1980to2023_final.csv")
df_songs = pd.read_csv("spotify_songs.csv")

print(df_books.head())
print(df_songs.head())

   Unnamed: 0         isbn                           title  \
0           0  9.78069E+12                   Summer Story    
1           1  9.78038E+12            The Lake of Darkness   
2           2  9.78035E+12  Beyond the Blue Event Horizon    
3           3  9.78045E+12               St. Peter's Fair    
4           4  9.78043E+12                       Twice Shy   

                    series_title series_release_number        authors  \
0                  Brambly Hedge                     2   Jill Barklem   
1                            NaN                   NaN   Ruth Rendell   
2                   Heechee Saga                     2  Frederik Pohl   
3  Chronicles of Brother Cadfael                     4   Ellis Peters   
4                            NaN                   NaN   Dick Francis   

                    publisher language  \
0                    Atheneum  English   
1  Vintage Crime/Black Lizard  English   
2            Ballantine Books  English   
3            Mysteri

### 1.1 Inspect the columns

Before cleaning the data, we inspect the column names to identify which fields are most useful for semantic retrieval.


In [3]:
print("Books dataframe columns:")
print(df_books.columns)

print("Songs dataframe columns:")
print(df_songs.columns)

Books dataframe columns:
Index(['Unnamed: 0', 'isbn', 'title', 'series_title', 'series_release_number',
       'authors', 'publisher', 'language', 'description', 'num_pages',
       'format', 'genres', 'publication_date', 'rating_score', 'num_ratings',
       'num_reviews', 'current_readers', 'want_to_read', 'price', 'url'],
      dtype='object')
Songs dataframe columns:
Index(['track_id', 'track_name', 'track_artist', 'lyrics', 'track_popularity',
       'track_album_id', 'track_album_name', 'track_album_release_date',
       'playlist_name', 'playlist_id', 'playlist_genre', 'playlist_subgenre',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'duration_ms', 'language'],
      dtype='object')


### 1.2 Select useful columns

Not every column is useful for retrieval. In this step, we keep only the columns that are most relevant for semantic search and recommendation quality.

For books, we keep title, author, language, genres, description, and rating-related fields.

For songs, we keep song title, artist, lyrics, genre, subgenre, language, and popularity.

In [4]:
# Keep only the most useful columns from the books dataset
df_books = df_books[
    [
        "title",
        "authors",
        "language",
        "description",
        "genres",
        "rating_score",
        "num_ratings"
    ]
].copy()

# Keep only the most useful columns from the songs dataset
df_songs = df_songs[
    [
        "track_name",
        "track_artist",
        "lyrics",
        "playlist_genre",
        "playlist_subgenre",
        "language",
        "track_popularity"
    ]
].copy()

# Check the remaining columns
print("Books columns after selection:")
print(df_books.columns)

print("\nSongs columns after selection:")
print(df_songs.columns)

Books columns after selection:
Index(['title', 'authors', 'language', 'description', 'genres', 'rating_score',
       'num_ratings'],
      dtype='object')

Songs columns after selection:
Index(['track_name', 'track_artist', 'lyrics', 'playlist_genre',
       'playlist_subgenre', 'language', 'track_popularity'],
      dtype='object')


### 1.3 Check missing values

Because semantic retrieval depends heavily on text, missing values in important columns can reduce the quality of recommendations.

In this step, we check for missing values before cleaning the datasets.

In [5]:
print("Missing values in books:")
print(df_books.isna().sum())

print("Missing values in songs:")
print(df_songs.isna().sum())

Missing values in books:
title            0
authors          0
language        26
description      2
genres           0
rating_score     0
num_ratings      0
dtype: int64
Missing values in songs:
track_name             0
track_artist           0
lyrics               260
playlist_genre         0
playlist_subgenre      0
language             260
track_popularity       0
dtype: int64


### 1.4 Clean the datasets

In this step, we clean both datasets before creating the searchable text.

For books and songs, we remove rows with missing key text fields because these fields are essential for semantic retrieval.

We also standardize the text formatting and remove duplicate entries to improve consistency and retrieval quality.

We keep the language column instead of dropping non-English entries, because language can later be used as an optional filter during retrieval.

In [6]:
#Cleaning Book Dataframe
# Remove rows with missing important text fields
# These columns are important because they provide the main semantic meaning
df_books = df_books.dropna(subset=["description"])

# Standardize text formatting
# astype(str) makes sure values are treated as text
# str.strip() removes extra spaces at the beginning or end
df_books["title"] = df_books["title"].astype(str).str.strip()
df_books["authors"] = df_books["authors"].astype(str).str.strip()
df_books["description"] = df_books["description"].astype(str).str.strip()
df_books["genres"] = df_books["genres"].astype(str).str.strip()

# Keep language as a usable field, but fill missing values instead of deleting rows
df_books["language"] = df_books["language"].fillna("Unknown").astype(str).str.strip()

# Remove duplicate books so retrieval does not repeat the same title
df_books = df_books.drop_duplicates(subset=["title", "authors"]).reset_index(drop=True)

print(df_books.isna().sum())


title           0
authors         0
language        0
description     0
genres          0
rating_score    0
num_ratings     0
dtype: int64


In [7]:
# Remove rows with missing important text fields
# Lyrics are especially important because they carry most of the meaning
df_songs = df_songs.dropna(subset=["lyrics"])

# Standardize text formatting
df_songs["track_name"] = df_songs["track_name"].astype(str).str.strip()
df_songs["track_artist"] = df_songs["track_artist"].astype(str).str.strip()
df_songs["lyrics"] = df_songs["lyrics"].astype(str).str.strip()
df_songs["playlist_genre"] = df_songs["playlist_genre"].astype(str).str.strip()
df_songs["playlist_subgenre"] = df_songs["playlist_subgenre"].astype(str).str.strip()

# Keep language as a usable field, but fill missing values instead of deleting rows
df_songs["language"] = df_songs["language"].fillna("Unknown").astype(str).str.strip()

# Remove duplicate songs so retrieval results are more diverse
df_songs = df_songs.drop_duplicates(subset=["track_name", "track_artist"]).reset_index(drop=True)

print(df_songs.isna().sum())

track_name           0
track_artist         0
lyrics               0
playlist_genre       0
playlist_subgenre    0
language             0
track_popularity     0
dtype: int64


### 1.5 Create combined text for retrieval

After cleaning the datasets, we create one combined text field for each row.

This step is important because embedding models work better when each item is represented as one richer text description instead of separate columns.

For books, we combine title, author, language, genres, description, and rating information.

For songs, we combine title, artist, language, genre, subgenre, lyrics, and popularity information.

In [8]:
# Combine important book columns into one structured text field
# This gives the embedding model richer context for semantic search
df_books["combined_text"] = (
    "Book Title: " + df_books["title"].fillna("").astype(str) + ". " +
    "Author: " + df_books["authors"].fillna("").astype(str) + ". " +
    "Language: " + df_books["language"].fillna("").astype(str) + ". " +
    "Genres: " + df_books["genres"].fillna("").astype(str) + ". " +
    "Description: " + df_books["description"].fillna("").astype(str) + ". " +
    "Rating Score: " + df_books["rating_score"].fillna(0).astype(str) + ". " +
    "Number of Ratings: " + df_books["num_ratings"].fillna(0).astype(str) + "."
)

# Preview one formatted book entry
print(df_books["combined_text"].iloc[0])

Book Title: Summer Story. Author: Jill Barklem. Language: English. Genres: ['Picture Books', 'Childrens', 'Fiction', 'Animals', 'Fantasy', 'Classics', 'Short Stories']. Description: It was such a hot summer. The sky was deep blue and the sun never faltered.All along Brambly Hedge, the mice did their best to keep cool. Poppy Eyebright sought refuge in the mossy shadows of the mill wheel; Dusty Dogwood took to walking by the banks of the cooling stream. Dusty and Poppy spent more and more time together, so no one was at all surprised when they announced their engagement.
They decided on a very unusual setting for the wedding ceremony, but even they didn't realize just how unusual it would prove to be!. Rating Score: 4.45. Number of Ratings: 1017.


In [9]:
# Combine important song columns into one structured text field
# Lyrics are especially important because they carry most of the meaning
df_songs["combined_text"] = (
    "Song Title: " + df_songs["track_name"].fillna("").astype(str) + ". " +
    "Artist: " + df_songs["track_artist"].fillna("").astype(str) + ". " +
    "Language: " + df_songs["language"].fillna("").astype(str) + ". " +
    "Genre: " + df_songs["playlist_genre"].fillna("").astype(str) + ". " +
    "Subgenre: " + df_songs["playlist_subgenre"].fillna("").astype(str) + ". " +
    "Lyrics: " + df_songs["lyrics"].fillna("").astype(str) + ". " +
    "Popularity: " + df_songs["track_popularity"].fillna(0).astype(str) + "."
)

# Using short_text for the model, to avoid overwhelming the model as it gets confused between the song lyrics and book titles 
# And hallucinates a lot because of how big the lyrics are 
# Note: Lyrics are still used in combined_text for embeddings and retrieval, just for the description the short_text is used
df_songs["short_text"] = (
    "Song Title: " + df_songs["track_name"].fillna("").astype(str) + ". " +
    "Artist: " + df_songs["track_artist"].fillna("").astype(str) + ". " +
    "Genre: " + df_songs["playlist_genre"].fillna("").astype(str) + ". " +
    "Subgenre: " + df_songs["playlist_subgenre"].fillna("").astype(str) + "."
)

# Preview one formatted song entry
print(df_songs["combined_text"].iloc[0])
print("Short Format:", df_songs["short_text"].iloc[0])

Song Title: Pangarap. Artist: Barbie's Cradle. Language: tl. Genre: rock. Subgenre: classic rock. Lyrics: Minsan pa Nang ako'y napalingon Hindi ko alam Na ika'y tutugon Sa mga tanong na aking nabitawan Hindi ko alam kung ito'y totoo Pangarap ka Sa bawat sandali Langit man ang tingin ko Sayo sana'y marating Hanggang dito na lang yata Ang kaya kong gawin Mangarap na lang At bumulong sa hangin Kailan kaya Darating ulit ang isang Sandali Na ako'y lilingon muli Pangarap ka o tinig mong kay lamig Ang iyong mga ngiti na sa akin ay Nakapagbigay pansin (Ikaw ba ay isang pangarap lang) Pangarap ka o tinig mong kay lamig Ang iyong mga ngiti Na sa akin ay Nakapagbigay... Pangarap ka o tinig mong kay lamig Ang iyong mga ngiti Na sa akin ay Nakapagbigay Pangarap ka o tinig mong kay lamig Ang iyong mga ngiti Na sa akin ay Nakapagbigay pansin. Popularity: 41.
Short Format: Song Title: Pangarap. Artist: Barbie's Cradle. Genre: rock. Subgenre: classic rock.


**Explanation:** 
For books we combined: title + author + language + genres + description + rating
For songs we combined: title + artist + language + genre + subgenre + lyrics + popularity

We decided to include lyrics for songs because it has the strongest relation to the song where it captures the emotional language, themes and mood that user would be looking for. For example, when user asks for a song that is heartbreaking and emotional it would match better in lyrics than the genre like pop. We also chose to include genre and subgenre for songs because they give additional context about the style and feel of the music. For example a classical love song and a pop love song. 

However, during testing we noticed the model getting overwhelemed with the length of the lyrics. There have been many cases where the song lyrics are being confused as book titles and put in the output. To avoid this we created short_text which will be used to display the description of the output. The LLM is still using the combined_text to for getting the embeddings and retrieval. 

We chose to include description and genres for books because book descriptions tell you what the book is actually about like the themes, and the feeling it gives you. This would be something user describes when they search for "a dark mysterious thriller" or "an uplifting love story." We included rating score and number of ratings for books because they show how well received a book is. A book with thousands of high ratings is more likely to be a reliable recommendation than one with very few.

We included track popularity for songs for the same reason, it helps surface songs that are well known and well received rather than obscure tracks that may not resonate with the user. We included language for both books and songs so the system can later be filtered by language if needed, giving the user more control over results.

We deliberately left out purely numeric columns like num_pages, duration_ms, key, loudness, and tempo because numbers alone carry no meaning that the embedding model can understand or use for matching.

## Step 2: Loading and generating embeddings

In this step, we load a pre-trained sentence transformer model to convert text into numerical vector representations (embeddings).

These embeddings allow the system to understand the meaning of text and compare similarity between user queries, books, and songs.

In [10]:
# Import the embedding model
from sentence_transformers import SentenceTransformer

# Load a lightweight and efficient model for semantic similarity
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully!


### 2.1 Generate embeddings

In this step, we convert the combined text fields from both datasets into embeddings.

These embeddings will be used to compare the user's query with the books and songs based on semantic meaning.

In [11]:
# Generate embeddings for the books dataset
book_embeddings = embedding_model.encode(
    df_books["combined_text"].tolist(),
    show_progress_bar=True
)

# Generate embeddings for the songs dataset
song_embeddings = embedding_model.encode(
    df_songs["combined_text"].tolist(),
    show_progress_bar=True
)

print(f"Created {len(book_embeddings)} embeddings, each with {book_embeddings[0].shape[0]} numbers")
print(f"Created {len(song_embeddings)} embeddings, each with {song_embeddings[0].shape[0]} numbers")

Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Batches:   0%|          | 0/511 [00:00<?, ?it/s]

Created 4156 embeddings, each with 384 numbers
Created 16342 embeddings, each with 384 numbers


## Step 3: Initialize and Loading ChromaDB and Retrival Functions 

In Lecture 8, retrieval was done using ChromaDB instead of manually calculating cosine similarity.

In this step, we initialize ChromaDB and create separate collections for books and songs so that both datasets can be searched efficiently.

In [18]:
# Import ChromaDB
import chromadb

# Create a Chroma client
client = chromadb.Client()

print("ChromaDB collections created successfully.")

ChromaDB collections created successfully.


### 3.1 Load Books and Songs into ChromaDB

In this step, we initialize ChromaDB and load both the books and songs collections into the database.

Each collection stores the combined text, embeddings, and metadata for its dataset. Metadata fields like `rating_score` and `track_popularity` are included so that results can be filtered later during retrieval.

Since the songs dataset has over 16,000 rows and ChromaDB has a maximum batch size of 5,461, the songs are added in chunks of 5,000 at a time.

In [17]:
# Delete if already exists so re-running doesn't crash
try:
    client.delete_collection(name="books_collection")
    client.delete_collection(name="songs_collection")
except:
    pass

books_collection = client.create_collection(name="books_collection")
songs_collection  = client.create_collection(name="songs_collection")

# Add Books
book_ids       = df_books.index.astype(str).tolist()
book_documents = df_books["combined_text"].tolist()
book_metadatas = df_books[["rating_score", "num_ratings"]].to_dict(orient="records")
book_emb_list  = book_embeddings.tolist()

# Books fit in one batch so no loop needed
books_collection.add(
    ids        = book_ids,
    documents  = book_documents,
    metadatas  = book_metadatas,
    embeddings = book_emb_list
)
print(f"Books loaded: {books_collection.count()}")

# Add Songs (needs batching — 16k rows exceeds ChromaDB's 5461 limit)
song_ids       = df_songs.index.astype(str).tolist()
song_documents = df_songs["combined_text"].tolist()
song_metadatas = df_songs[["track_popularity"]].to_dict(orient="records")
song_emb_list  = song_embeddings.tolist()

for start in range(0, len(song_ids), 5000):
    end = start + 5000
    songs_collection.add(
        ids        = song_ids[start:end],
        documents  = song_documents[start:end],
        metadatas  = song_metadatas[start:end],
        embeddings = song_emb_list[start:end]
    )

print(f"Songs loaded: {songs_collection.count()}")

Books loaded: 4156
Songs loaded: 16342


### 3.2 Retrieve Relevant Records

In this step, we define the retrieval function that searches both the books and songs collections in ChromaDB.

Given a user query, the function converts it into an embedding using the same model used to encode the datasets, then searches both collections for the most semantically similar results.

It returns the matched descriptions as formatted strings ready to be passed into the LLM prompt, along with the corresponding dataframe rows for display.

In [19]:
def retrieve_relevant_records(query, k=5, book_filter=None, song_filter=None):
    query_embedding = embedding_model.encode([query])

    book_results = books_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        where=book_filter if book_filter else None
    )

    song_results = songs_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        where=song_filter if song_filter else None
    )

    book_ids = [int(i) for i in book_results["ids"][0]]
    song_ids = [int(i) for i in song_results["ids"][0]]

    book_rows = df_books.loc[book_ids]
    song_rows = df_songs.loc[song_ids]

    book_context = "\n\n".join(book_results["documents"][0])
    song_context = "\n\n".join(df_songs.loc[song_ids]["short_text"].tolist()) #Uses the short_text based of the song_ids to get the song context to not overwhelm the LLM

    return book_context, song_context, book_rows, song_rows

### 3.2.1 Testing the Retrieval

In [20]:
book_context, song_context, book_rows, song_rows = retrieve_relevant_records(
    "a sad love story",
    k=5,
    book_filter={"rating_score": {"$gte": 4.0}},
    song_filter=None
)

In [21]:
# Look at the top 5 picks
print("MATCHED BOOKS")
display(book_rows[["title", "authors", "rating_score"]])

print("MATCHED SONGS")
display(song_rows[["track_name", "track_artist", "track_popularity"]])

MATCHED BOOKS


,title,authors,rating_score
3298,Love's Suicide,Jennifer Foor,4.07
1765,The Love of a Good Woman,Alice Munro,4.01
3211,Crashed,K. Bromberg,4.52
4072,Hello Beautiful,Ann Napolitano,4.20
1815,A Walk to Remember,Nicholas Sparks,4.20


MATCHED SONGS


,track_name,track_artist,track_popularity
5765,Moral of the Story,Ashe,53
7758,I Wish,Carl Thomas,52
2797,Faded Pictures,Case,58
5058,Faded Pictures - From The Rush Hour Soundtrack,Case,0
5991,Dear Friends - Remastered 2011,Queen,32


---
## Part 4: Augmentation (Connecting to the LLM)

We now have the 5 most relevant books and songs descriptions. The final step is to inject them into a prompt and send it to Ollama.

Below, we have tested the different LLM models and decided which would work best for our recommendation system. We also tested different versions of prompts and designed the final prompt with guardrails that will shape how the user's query is sent to the AI.

### 4.1 Testing the Models

In [22]:
ollama.pull('llama3.2:3b')

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [23]:
ollama.pull('deepseek-r1:1.5b')

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [24]:
# Testing Qwen model
response = ollama.generate(
    model='qwen2.5:1.5b',
    prompt="""
I want 2 books and 2 songs that are romantic. 

Format:

Books:
1.
2.

Songs:
1.
2.
""",
    options={'temperature': 0.1}
)

print("Qwen Output:\n")
print(response['response'])

Qwen Output:

Certainly! Here are two romantic book suggestions along with two romantic song recommendations for you to consider:

### Books:
1. "Pride and Prejudice" by Jane Austen - A classic novel about love, marriage, and the social norms of 19th-century England.
2. "The Notebook" by Nicholas Sparks - This heartwarming story follows a young couple's journey across America as they fall in love.

### Songs:
1. "Someone Like You" by Adele - A timeless song that perfectly captures the pain of lost love and the hope for redemption.
2. "Thinking Out Loud" by Ed Sheeran - This beautiful ballad explores themes of heartbreak, self-reflection, and finding strength within oneself.

These selections should provide a romantic reading and listening experience to enhance your mood!


In [25]:
# Testing Llama model

response = ollama.generate(
    model='llama3.2:3b',
    prompt="""
I want 2 books and 2 songs that are romantic. 

Format:

Books:
1.
2.

Songs:
1.
2.
""",
    options={'temperature': 0.1}
)

print("Llama Output:\n")
print(response['response'])

Llama Output:

Here are two romantic book recommendations and two romantic song suggestions for you:

Books:
1. "The Notebook" by Nicholas Sparks - A classic romance novel about the love story of two young souls, Noah and Allie, who are separated by social class but find their way back to each other years later.
2. "Me Before You" by Jojo Moyes - A heart-wrenching love story about Louisa, a quirky caregiver, and Will, a wealthy young man paralyzed in an accident, as they navigate the complexities of love, loss, and life.

Songs:
1. "A Thousand Years" by Christina Perri - A beautiful ballad that captures the essence of eternal love, with lyrics that speak to the idea of finding your soulmate and spending forever together.
2. "Perfect" by Ed Sheeran - A romantic acoustic song about finding the perfect person, with lyrics that celebrate the beauty of love and relationships.

I hope you enjoy these recommendations!


In [26]:
# Testing DeepSeek model

response = ollama.generate(
    model='deepseek-r1:1.5b',
    prompt="""
I want 2 books and 2 songs that are romantic. 

Format:

Books:
1.
2.

Songs:
1.
2.
""",
    options={'temperature': 0.1}
)

print("DeepSeek Output:\n")
print(response['response'])

DeepSeek Output:

**Books:**

1. **"The Art of the Love Triangle" by Michael Connelly**
   - This book delves deeply into the study of relationships, offering insights into love, loss, and finding oneself. It is highly regarded for its exploration of emotional themes in relationships.

2. **"I Will Always Be Your Love" by Taylor Swift**
   - A heartfelt track that explores themes of enduring commitment and self-acceptance, it has become a classic example of emotional resonance and powerful lyrics.

**Songs:**

1. **"I Will Always Be Your Love" by Taylor Swift**
   - A song that resonates with strong emotional hooks, particularly its focus on love, commitment, and the enduring nature of relationships.

2. **"You're the Best" by Taylor Swift**
   - A powerful track about self-love and self-acceptance, it has become a staple in romantic songs for its emotional depth and message.

These selections capture both the themes of romance and emotional depth, making them suitable as recommendatio

**Model Evaluation**: 
Query Model: 
This model was the quickest to produce the output. It followed the format well which was displaying 2 books and 2 songs. The explanations of the books and songs were short and concise explanation which made it well readable. 

Llama Model:
This model also followed the format well. The explanation for the books and songs were more natural and detailed. Howvever, there was an extra wording added "Here are my reccomendations" which shows the model was less strict in following instructions.  

DeepSeek Model: 
This model has the format, but the output given was weak and less reliable reccomendations. There was also no explanation of the books and songs chosen which was something expected. 

Final Decision: The model we choose to follow is the Query Model (Qwen2.5:1.5b) because it gave the most consistent, structured, and well-presented output for our recommendation system and it is small enough to run smoothly on a laptop which is important for our live demo.

### 4.2 Testing the Prompts

In [27]:
#Prompt V1 
response = ollama.generate(
    model='qwen2.5:1.5b',
    prompt='I want motivational books and songs with an uplifting and hopeful vibe.',
    options={'temperature': 0.1}
)

print("Prompt First Version Testing:\n")
print(response['response'])

Prompt First Version Testing:

Certainly! Here's a list of some inspirational books and songs that can help you feel uplifted and hopeful:

### Books:
1. **"Man's Search for Meaning" by Viktor E. Frankl**
   - This book is about the author’s experiences in Nazi concentration camps, where he found meaning through his work as a psychiatrist.

2. **"The Alchemist" by Paulo Coelho**
   - A heartwarming story of a shepherd boy who follows his dreams and discovers that his destiny is to find the treasure hidden in the desert.

3. **"The Power of Now: A Guide to Present-Moment Living" by Eckhart Tolle**
   - This book teaches you how to live fully in the present moment, which can help reduce stress and increase happiness.

4. **"The Alchemist's Journey" by Paulo Coelho**
   - Another novel from Paulo Coelho that explores themes of following your dreams and finding your purpose.

5. **"The Four Agreements" by Don Miguel Ruiz**
   - This book provides practical guidelines for living a more peac

**Explanation:** This first version of the prompt produces an output that gives relevant reccomendations but it based on general knowledge rather than giving output based on the retireved data. The output also lacks structure, there is too many books and songs that are genertated with unncessary details. To improve the prompt the next version should incorporate clearer formatting instructions and contrains like specifying the exact number of reccomendations and limit the length of the response. 

In [28]:
# Prompt Version 2 Testing: Making the prompt to have structure
response = ollama.generate(
    model='qwen2.5:1.5b',
    prompt="""
You are a recommendation assistant.

User request:
I want motivational books and songs with an uplifting and hopeful vibe.

Instructions:
- Recommend at least 2 books and 2 songs
- Keep the response concise
- Briefly explain why each recommendation matches the request
- Clearly separate books and songs

Format:

Books:
1.
2.

Songs:
1.
2.
""",
    options={'temperature': 0.1}
)

print("Prompt Version 2 Testing:\n")
print(response['response'])

Prompt Version 2 Testing:

Books:
1. "Man's Search for Meaning" by Viktor E. Frankl - This book offers profound insights into finding meaning in life, even during difficult times, which can be very uplifting and hopeful.
2. "The Alchemist" by Paulo Coelho - This novel encourages following your dreams and the path that feels right to you, offering a sense of hope and motivation.

Songs:
1. "Hopelessly in Love" by The Script - This song's lyrics are full of optimism and can inspire listeners to keep their hopes alive.
2. "Don't Stop Believin'" by Journey - This classic rock anthem is known for its uplifting message of never giving up on your dreams, which can be very motivational.


**Explanation:** The second version of the prompt is better than the first prompt as there is clear instructions and structure. It follows the required format and returns exactly 2 books and 2 songs. The responses are more organized and concise which makes it more readable. However, there still needs to be improvements as the model is still relying on general knowledge than the retrieved data. 

In [29]:
#Prompt Version 3 Testing: Prompt has structure but includes RAG and guardrails
query = "I want motivational books and songs with an uplifting and hopeful vibe."

# Step 1: Retrieve relevant records
book_context, song_context, book_rows, song_rows = retrieve_relevant_records(query, k=3)


# Step 2: Handle empty retrieval
if len(book_context.strip()) == 0:
    book_context = "empty"

if len(song_context.strip()) == 0:
    song_context = "empty"

# Step 3: Send retrieved context into the prompt
response = ollama.generate(
    model='qwen2.5:1.5b',
    prompt=f"""
You are a book and music recommendation assistant.

User query:
{query}

Retrieved Books:
{book_context}

Retrieved Songs:
{song_context}

Rules:
- ONLY use the retrieved books and songs provided above
- Do not make up any books, authors, songs, or artists
- Recommend EXCATLY 2 books and 2 songs
- Keep the response concise, minimum 3 sentences
- Briefly explain why each recommendation matches the user's query 
- If the retrieved context is weak or not relevant, say no match is found

""",
    options={'temperature': 0.1}
)

print("Prompt Version 3 Testing:\n")
print(response['response'])

Prompt Version 3 Testing:

Based on your preference for motivational books and uplifting songs with a hopeful vibe, I recommend:

1. **Maybe Someday** by Colleen Hoover - This book combines romance with contemporary themes, offering hope and resilience through its story of music and love. The genre mix includes romance, new adult, and young adult, which aligns well with your interests.

2. **Out of the Dust** by Karen Hesse - Set against the backdrop of the Great Depression, this historical fiction explores themes of loss, resilience, and hope. It's a powerful story that resonates with its realistic portrayal of hardship and the strength it takes to overcome adversity.

3. **Transcendent Kingdom** by Yaa Gyasi - This novel delves into the complexities of family dynamics, faith, science, and mental health. The genre mix includes literary fiction and contemporary literature, making it a rich source for exploring personal growth and overcoming challenges.

4. **Delight** by Solitary Exper

**Explanation:** This version improves the output as it uses RAG to retrive relevant book and song context from the datasets. As a result, the model begins to generate recommendations that are more aligned with the retrieved data while still remaining concise and follows the required format. 

However, the prompt does not fully enforce the use of the retrieved results. The model may still rely on its general knowledge as it did not strictly select from the top retrieved record when I tried testing. This means there is a need for a stronger prompt and guardrails to ensure the output is more reliable. 

### 4.3 Final Version of Prompting and Guardrails 

In [30]:
def generate_answer(query, book_context, song_context):
    """
    Given a user query and retrieved book and song descriptions,
    ask the LLM to generate a recommendation.
    """
    prompt = f"""
    You are Readify, a book and music recommendation assistant to help find great books and songs.

    Based ONLY on the following retrieved books and songs, answer the user's request.
    As long as the request refers to books or music more generally, and RETRIEVED BOOKS and RETRIEVED SONGS are not "empty", at least mention one book from RETRIEVED BOOKS and one song from RETRIEVED SONGS as a recommendation.
    Always start your response with a warm and exciting opening line like "Great choice!" or "I have the perfect recommendations for you!" before giving the recommendations.
    Write in a friendly and engaging way and add at least two sentences, maximum three sentences, explaining why each recommended book and song is relevant to the request.
    Always separate your response into two sections Books: and then Songs: 
    You are allowed to say "No match found" but only as a last resort if the retrieved books and songs do not match the request.
    Do NOT make up any book titles, author names, song titles, or artist names that are not in the retrieved lists below.
    Do NOT copy or repeat any of the retrieved descriptions, lyrics, or metadata in your response.
    
    RETRIEVED BOOKS:
    {book_context}

    RETRIEVED SONGS:
    {song_context}

    USER REQUEST: {query}
    """

    response = ollama.generate(
        model= "qwen2.5:1.5b",
        prompt=prompt,
        options={'temperature': 0}  # Lower temperature = more focused, less random
    )

    return response['response']

In [31]:
def rag_query(query, k=5, book_filter=None, song_filter=None):
    """
    The complete RAG pipeline: Retrieve → Augment → Generate.
    Returns the LLM's answer, matched book rows, and matched song rows.
    """
        
    # Step 1: Retrieve the most relevant books and songs
    book_context_raw, song_context_raw, book_rows, song_rows = retrieve_relevant_records(query, k, book_filter, song_filter)

    # Step 2: Build the context for the LLM with empty fallback 
    if len(query.strip()) == 0:
        return "Please enter a query to get recommendations!", empty_books, empty_songs
        
    if len(book_context_raw.strip()) == 0:
        book_context = "empty"
    else:
        book_context = book_context_raw

    if len(song_context_raw.strip()) == 0:
        song_context = "empty"
    else:
        song_context = song_context_raw
        
    # Step 3: Generate the answer
    answer = generate_answer(query, book_context, song_context)
    
    return answer, book_rows, song_rows

---
## Part 5: Final Product

Let's test the full pipeline. For each query, we'll see:
- The LLM's conversational recommendation
- The DataFrame of matched books
- The DataFrame of matched songs

**Filters available:**
- `$gte`: greater than or equal to
- `$lte`: less than or equal to
- `$eq`: equal to
- `$and`: combine multiple filters

In [32]:
# Demo 1: Romantic and emotional query with no filters
query = "Can you suggest a romantic and emotional book and a song that would make me cry?"
answer, book_matches, song_matches = rag_query(
    query,
    k=5
)
print("Recommendation:\n", answer)
print("\nTop book matches:")
display(book_matches[["title", "authors", "rating_score"]])
print("\nTop song matches:")
display(song_matches[["track_name", "track_artist", "track_popularity"]])

Recommendation:
 Great choice! I have the perfect recommendations for you!

**Books:**
I recommend "Bleeding Love" by Harper Sloan. This contemporary romance novel follows Liam Beckett on his mission to prove to Katy Michaels that love is worth fighting for, even after a heart-wrenching past. The story explores themes of loss and healing, making it an emotionally resonant read.

**Songs:**
For the song recommendation, I suggest "Tears In The Rain" by The Weeknd. This R&B track has a powerful emotional impact that can make you feel tears well up in your eyes. Its lyrics about raindrops falling on broken hearts perfectly capture the feeling of heartbreak and longing for love.

Both books and songs are designed to evoke strong emotions, making them perfect choices for when you need a moment to let yourself be touched by something truly beautiful or heartbreaking. Enjoy!

Top book matches:


,title,authors,rating_score
3340,Consolation,Corinne Michaels,4.33
3298,Love's Suicide,Jennifer Foor,4.07
3393,Bleeding Love,Harper Sloan,4.23
3943,All Rhodes Lead Here,Mariana Zapata,4.31
4072,Hello Beautiful,Ann Napolitano,4.20



Top song matches:


,track_name,track_artist,track_popularity
10363,Tears In The Rain,The Weeknd,43
5519,Love Will Tear Us Apart,Joy Division,71
5968,All Cried Out (with Full Force),Lisa Lisa & Cult Jam,36
3945,Love Will Tear Us Apart - 2010 Remaster,Joy Division,64
3777,Tears,The Tragic Thrills,0


In [33]:
# Demo 2: Same query with books rated equivalent or higher than a 4.0 and songs that have a 70 rating popularity 
# We noticed Demo 1 returned songs with very low popularity, so we add a song popularity filter to fix this
query = "Can you suggest a romantic and emotional book and a song that would make me cry?"
answer, book_matches, song_matches = rag_query(
    query,
    k=5,
    book_filter={"rating_score": {"$gte": 4.0}},
    song_filter={"track_popularity": {"$gte": 70}}
)
print("Recommendation:\n", answer)
print("\nTop book matches:")
display(book_matches[["title", "authors", "rating_score"]])
print("\nTop song matches:")
display(song_matches[["track_name", "track_artist", "track_popularity"]])

Recommendation:
 Great choice! I have the perfect recommendations for you!

**Books:**
I recommend "Bleeding Love" by Harper Sloan. This contemporary romance series follows Liam Beckett, who is on a mission to prove to Katy Michaels that love worth having is a love worth fighting for. The story explores themes of loss and healing as Katy navigates her past heartbreaks while falling in love with Liam again. It's a beautifully written book that will make you feel your heart ache.

**Songs:**
For the song, I suggest "Love Will Tear Us Apart" by Joy Division. This classic electropop track is known for its powerful lyrics and haunting melody. The song perfectly captures the emotional depth of love and loss, making it an excellent choice to accompany a romantic story that will make you cry.

Both books and songs are designed to evoke strong emotions, so they should be perfect companions as you delve into your new favorite reads and listen to some beautiful music. Enjoy!

Top book matches:


,title,authors,rating_score
3340,Consolation,Corinne Michaels,4.33
3298,Love's Suicide,Jennifer Foor,4.07
3393,Bleeding Love,Harper Sloan,4.23
3943,All Rhodes Lead Here,Mariana Zapata,4.31
4072,Hello Beautiful,Ann Napolitano,4.20



Top song matches:


,track_name,track_artist,track_popularity
5519,Love Will Tear Us Apart,Joy Division,71
8041,Big Girls Don't Cry (Personal),Fergie,74
6285,You should be sad,Halsey,86
3046,Cry for Me,Camila Cabello,70
12106,Total Eclipse of the Heart,Bonnie Tyler,74


In [34]:
# Demo 3: Dark and mysterious with a 50 song rating 
query = "I want something dark and mysterious with a tense atmosphere"
answer, book_matches, song_matches = rag_query(
    query,
    k=5,
    song_filter={"track_popularity": {"$gte": 50}}
)
print("Recommendation:\n", answer)
print("\nTop book matches:")
display(book_matches[["title", "authors", "rating_score"]])
print("\nTop song matches:")
display(song_matches[["track_name", "track_artist", "track_popularity"]])

Recommendation:
 Great choice! Here are some recommendations that fit your request for something dark, mysterious, and with a tense atmosphere:

**Books:**
I have the perfect recommendations for you! The book "Darker After Midnight" by Lara Adrian is an excellent choice. It's part of her bestselling Midnight Breed series and features a thrilling vampire romance set against the backdrop of a world on the edge of chaos. The story explores themes of power, redemption, and the struggle between good and evil, making it both captivating and thought-provoking.

**Songs:**
For your musical taste, "Sometimes" by Miami Horror is an excellent choice. This pop-electropop song has a dark and mysterious vibe that fits perfectly with what you're looking for. The lyrics are haunting and the music adds to the overall tense atmosphere, making it perfect for when you need something more than just light-hearted tunes.

Both of these recommendations should provide a great experience that matches your prefe

,title,authors,rating_score
2438,Blue Noon,Scott Westerfeld,3.86
615,This Present Darkness,Frank E. Peretti,4.23
3055,Darker After Midnight,Lara Adrian,4.37
974,Nightfall,Isaac Asimov,4.05
1960,His Dark Materials,Philip Pullman,4.29



Top song matches:


,track_name,track_artist,track_popularity
12122,Sometimes,Miami Horror,51
7400,Pompeii,Bastille,70
14154,Fear Of The Dark - 1998 Remastered Version,Iron Maiden,55
5592,Pompeii - Audien Remix,Bastille,55
14476,It's Happening Again,Agnes Obel,52


In [35]:
# Demo 4: Specific book based reccomendation with a romantic and elegant vibe
query = "Recommend books and songs similar to Pride and Prejudice with a romantic and elegant vibe"
answer, book_matches, song_matches = rag_query(
    query,
    k=5,
    book_filter={"rating_score": {"$gte": 4.0}},
    song_filter={"track_popularity": {"$gte": 50}}
)
print("Recommendation:\n", answer)
print("\nTop book matches:")
display(book_matches[["title", "authors", "rating_score"]])
print("\nTop song matches:")
display(song_matches[["track_name", "track_artist", "track_popularity"]])

Recommendation:
 Great choice! I have the perfect recommendations for you!

For the user's request of recommending books and songs that are similar to "Pride and Prejudice" with a romantic and elegant vibe, here are my suggestions:

Books:
1. **The Book Thief** by Markus Zusak - This novel is set in Nazi Germany during World War II but focuses on the life of Liesel Meminger, who becomes an orphaned child at just 6 years old. The story is filled with romance, adventure, and a touch of tragedy that resonates deeply with readers.

Songs:
1. **"Beautiful Day" by U2** - This iconic song from their album "The Joshua Tree" perfectly captures the romantic and elegant vibe you're looking for. It's a timeless classic that has been featured in countless films and TV shows, adding to its enduring popularity.
   
Both books and songs offer a blend of romance, tragedy, and elegance that would complement your request beautifully!

Top book matches:


,title,authors,rating_score
3242,Beauty from Love,Georgia Cates,4.12
235,Annie on My Mind,Nancy Garden,4.02
3777,"The Cherries: Faith, Hope, Happiness. Does she...",D.B. Carter,4.58
278,A Rose in Winter,Kathleen E. Woodiwiss,4.18
132,My Torin,K. Webster,4.20



Top song matches:


,track_name,track_artist,track_popularity
9391,Beauty & Essex (feat. Daniel Caesar & Unknown ...,Free Nationals,60
14421,The Book of Love,The Magnetic Fields,60
2644,Pride and Joy,Stevie Ray Vaughan,65
10101,Humble And Kind,Tim McGraw,70
882,Beautiful Day,U2,72


In [36]:
#Demo 5: User specifies 3 books and 2 songs to reccomend based on dark, emotional, thriller theme
query = "I want 3 books and 2 songs that are dark and emotional and thrilling?"  
answer, book_matches, song_matches = rag_query(
    query,
    k=10,
    book_filter={"rating_score": {"$gte": 2.0}},
    song_filter={"track_popularity": {"$gte": 60}},
)
print("Recommendation:\n", answer)
print("\nTop book matches:")
display(book_matches[["title", "authors", "rating_score"]])
print("\nTop song matches:")
display(song_matches[["track_name", "track_artist", "track_popularity"]])

Recommendation:
 Great choice! Here are three books and two songs that perfectly capture the essence of darkness, emotion, and thrill:

**Books:**
1. **His Dark Materials by Philip Pullman** - This trilogy is a modern fantasy classic series that explores themes of love, loss, and redemption in a world where magic exists alongside science. The story follows Lyra and Will as they navigate through haunted otherworlds, encountering witches, armored bears, fallen angels, and soul-eating specters. It's a thrilling adventure that will leave you questioning everything you know about your world.
2. **The Drawing of the Three by Stephen King** - This novel is a masterful interweaving of dark fantasy with icy realism. Roland, the last gunslinger, encounters three mysterious doorways on the beach, each leading to a different person's life in contemporary New York. The story is filled with suspense and unexpected twists that will keep you on the edge of your seat.
3. **Soul Music by Terry Pratchett

,title,authors,rating_score
1960,His Dark Materials,Philip Pullman,4.29
2845,Beautiful Darkness,Kami Garcia,3.83
673,The Drawing of the Three,Stephen King,4.23
3228,Rare and Precious Things,Raine Miller,4.13
1332,Soul Music,Terry Pratchett,4.05
2394,The History of Love,Nicole Krauss,3.92
2502,The Book of Lost Things,John Connolly,3.96
1605,Exile's Song,Marion Zimmer Bradley,4.04
2417,Touching Darkness,Scott Westerfeld,3.90
961,Four Past Midnight,Stephen King,3.95



Top song matches:


,track_name,track_artist,track_popularity
7604,The Vengeful One,Disturbed,65
6675,Shot in the Dark,Ozzy Osbourne,62
2348,The Violence,Rise Against,62
13691,Dark Times,The Weeknd,69
4049,Darkside,Alan Walker,71
15130,Late Night Feelings (feat. Lykke Li),Mark Ronson,69
11639,Should Have Known Better,Sufjan Stevens,65
12866,Invisible - from the Netflix Film Klaus,Zara Larsson,70
11696,Addicted To Love,Robert Palmer,67
10126,Beautiful Creatures (feat. MAX),ILLENIUM,64


---
## Part 6: Building a Web Interface with Gradio

Right now, using the system requires running notebook cells. That's fine for development, but you wouldn't ask a customer to do that.

**Gradio** is a free Python library  that wraps any Python function in a simple web interface. It even generates a public link you can share with anyone.

In [37]:
import gradio as gr

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Lato:wght@300;400;700&display=swap');

.gradio-container {
    background: linear-gradient(135deg, #fe6a93, #343344, #01b66b) !important;
    min-height: 100vh !important;
    font-family: 'Lato', sans-serif !important;
}
.block, .gr-box, .gr-panel, .gr-form {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
}
textarea {
    background: rgba(255, 255, 255, 0.2) !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
    font-family: 'Lato', sans-serif !important;
    font-size: 16px !important;
}
textarea::placeholder {
    color: rgba(255, 255, 255, 0.6) !important;
}
button {
    background: rgba(255, 255, 255, 0.2) !important;
    color: white !important;
    border: 1px solid rgba(255, 255, 255, 0.3) !important;
    border-radius: 8px !important;
    font-family: 'Lato', sans-serif !important;
    font-size: 18px !important;
    font-weight: 700 !important;
}
table {
    background: rgba(0, 0, 0, 0.4) !important;
    border-radius: 8px !important;
    font-family: 'Lato', sans-serif !important;
    font-size: 15px !important;
}
thead tr th {
    background: rgba(0, 0, 0, 0.6) !important;
    color: white !important;
    border-bottom: 1px solid rgba(255, 255, 255, 0.2) !important;
}
tbody tr td {
    background: rgba(0, 0, 0, 0.3) !important;
    color: white !important;
    border-bottom: 1px solid rgba(255, 255, 255, 0.1) !important;
}
h1 {
    font-family: 'Playfair Display', serif !important;
    font-size: 42px !important;
    font-weight: 700 !important;
    color: white !important;
}
h2, h3 {
    font-family: 'Playfair Display', serif !important;
    font-size: 24px !important;
    color: white !important;
}
h4, p, label, span {
    font-family: 'Lato', sans-serif !important;
    font-size: 16px !important;
    color: white !important;
}
footer {
    display: none !important;
}
"""

def gradio_rag(query, min_rating, min_popularity, k):
    """
    Wrapper that connects our RAG pipeline to the Gradio interface.
    """
    # Empty query check
    if len(query.strip()) == 0:
        return "Please enter a query to get recommendations!", None, None

    book_filter = {"rating_score": {"$gte": float(min_rating)}}
    song_filter = {"track_popularity": {"$gte": int(min_popularity)}}

    answer, book_rows, song_rows = rag_query(
        query=query,
        k=int(k),
        book_filter=book_filter,
        song_filter=song_filter
    )

    book_display = book_rows.reset_index(drop=True)[["title", "authors", "rating_score", "genres"]]
    song_display = song_rows.reset_index(drop=True)[["track_name", "track_artist", "track_popularity", "playlist_genre"]]

    return answer.strip(), book_display, song_display


with gr.Blocks(title="Readify - Book & Music Recommender", css=custom_css ) as demo:
    gr.Markdown(
        """
        # 📚🎵 Readify
        ### Your personal book and music recommender
        Tell us what you are in the mood for and we will find the perfect books and songs for you.
        """
    )

    with gr.Row():
        query_box = gr.Textbox(
            label="What are you in the mood for?",
            placeholder="e.g. I want a book and a song that is romantic and emotional? ",
            lines=2,
        )

    with gr.Row():
        min_rating_slider = gr.Slider(
            minimum=float(df_books["rating_score"].min()),
            maximum=float(df_books["rating_score"].max()),
            value=3.5,
            step=0.1,
            label="Minimum Book Rating (out of 5)",
        )
        min_popularity_slider = gr.Slider(
            minimum=int(df_songs["track_popularity"].min()),
            maximum=int(df_songs["track_popularity"].max()),
            value=50,
            step=1,
            label="Minimum Song Popularity (out of 100)",
        )

    with gr.Row():
        k_slider = gr.Slider(
            minimum=1,
            maximum=10,
            value=5,
            step=1,
            label="Top-k results to retrieve",
        )
        
    run_button = gr.Button("🔍 Find my books and songs!")

    gr.Markdown("### ✨ Recommendation")
    answer_box = gr.Markdown(label="Recommendation")

    gr.Markdown("### 📚 Matched Books")
    books_table = gr.Dataframe(
        headers=["Title", "Author", "Rating", "Genres"],
        datatype=["str", "str", "number", "str"],
        label="Matched Books",
        interactive=False,
    )

    gr.Markdown("### 🎵 Matched Songs")
    songs_table = gr.Dataframe(
        headers=["Song", "Artist", "Popularity", "Genre"],
        datatype=["str", "str", "number", "str"],
        label="Matched Songs",
        interactive=False,
    )

    run_button.click(
        fn=gradio_rag,
        inputs=[query_box, min_rating_slider, min_popularity_slider, k_slider],
        outputs=[answer_box, books_table, songs_table],
    )

    gr.Markdown(
        """
        **Tip:** Adjust the sliders to broaden or narrow your results.
        The more specific your query, the better the recommendations!
        """
    )
# Launch the app — share=True creates a public link
demo.launch(share=False)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---
**What Can Go Wrong? Evaluating Your System**

Building the tool is only half the job. Here are common failure modes to watch for:

**1. Retrieval can miss.** Embedding similarity is approximate. A search for "a movie about losing your memory" might also return movies about Alzheimer's or head injuries, related but not what the user meant.

**2. The LLM can hallucinate.** Even though we tell it to use *only* the provided descriptions, small models like qwen2.5:1.5b may invent actor names or plot details that aren't in the context.

**3. Prompt design matters.** Small wording changes in the prompt can dramatically change output quality. If you don't tell the model to say "I don't know" when unsure, it will confidently make things up.

**What Worked Well:**
1. Queries that are based of mood and feelings work really well. For example, I want something romantic and emotional. The embedding model is very good at matching emotional language in descriptions and lyrics.
2. Filters improved results significantly. By adding a book rating filter and song popularity filter it helped get higher quality recommendations. For example adding song_filter with track_popularity above 40 removed songs with popularity of 0 and replaced them with more well known tracks.
3. Queries referring a certain book or a song and to reccomend a similar mood or theme work really well. 
4. When the user submits an empty query the system returned a clear warning message instead of returning random results.

**What Didn't Work Well:**
1. When passing full lyrics to the LLM it caused hallucination. The LLM would often confuse certain words in the lyrics as book titles and passed songs as the output. The short_text was the fix as the model was overwhelemd with full song lyrics and often the books output hallucinated. The model still uses the song combined text for embeddings and retrieval, but when displaying the output it uses the short_text. 
2. Complex queries caused hallucination. When the query had too many specific requirements at once like "uplifting and motivational with the highest rating going through a hard time", the model got confused and hallucinated author names and titles. So, simpler mood based queries worked much better.
3. The model does not always follow exact counts when specified in the query. For example when the user asked for 3 books and 2 songs the 
model sometimes gave 2 books instead of 3. This is because managing two counts across two lists is more complex than the single dataset. This shows that small models can be inconsistent with strict counting instructions.